# Nifty 500 Candlestick ML - Exploration Notebook

End-to-end walkthrough: data ingestion, preprocessing, pattern detection, feature engineering, model training, and signal generation.

In [ ]:
import sys, os
os.chdir('..')
sys.path.insert(0, '.')

In [ ]:
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

print('Config loaded:', list(cfg.keys()))

## 1. Data Ingestion

In [ ]:
from src.data_ingestion import download_universe, _build_date_range, NIFTY500_TICKERS

start, end = _build_date_range(cfg['data']['lookback_years'])
print(f'Date range: {start} to {end}')

SAMPLE_TICKERS = NIFTY500_TICKERS[:5]
raw_dir = Path(cfg['data']['raw_dir'])
raw_data = download_universe(SAMPLE_TICKERS, start, end, raw_dir)
print(f'Downloaded {len(raw_data)} tickers')

## 2. Preprocessing

In [ ]:
from src.preprocessing import preprocess_universe

processed_dir = Path(cfg['data']['processed_dir'])
processed = preprocess_universe(raw_data, processed_dir)

ticker = list(processed.keys())[0]
df = processed[ticker]
print(f'{ticker}: {df.shape}')
df.tail()

## 3. Pattern Detection

In [ ]:
from src.pattern_detection import detect_all_patterns

patterns = detect_all_patterns(df)
print('Pattern counts:')
print(patterns.sum().sort_values(ascending=False).head(10))

plt.figure(figsize=(12,4))
patterns.sum().sort_values(ascending=False).plot(kind='bar', color='steelblue', edgecolor='white')
plt.title(f'Pattern Occurrences - {ticker}')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
from src.feature_engineering import build_feature_matrix

feat_df = build_feature_matrix(df, cfg).dropna()
print(f'Feature matrix: {feat_df.shape}')
feat_df.describe().T.head(20)

## 5. Label Generation

In [ ]:
from src.signals import add_labels

labeled = add_labels(feat_df, cfg)
print('Label distribution:')
print(labeled['label'].value_counts(normalize=True).mul(100).round(2))

## 6. Model Training

In [ ]:
from src.model import get_feature_cols, time_split, train_with_cv, evaluate_model

feature_cols = get_feature_cols(labeled)
train_df, test_df = time_split(labeled, train_months=cfg['data']['train_split_months'], test_months=cfg['data']['test_split_months'])

print(f'Train: {len(train_df)} | Test: {len(test_df)} | Features: {len(feature_cols)}')

models = train_with_cv(train_df, feature_cols, label_col='label', cfg=cfg, n_splits=3)

## 7. Evaluation

In [ ]:
X_test = test_df[feature_cols].astype(float)
y_test = test_df['label']

for name, model in models.items():
    metrics = evaluate_model(model, X_test, y_test, threshold=cfg['model']['confidence_threshold'], model_name=name)
    print(metrics)

## 8. SHAP Feature Importance

In [ ]:
from src.model import compute_shap
import shap

shap_vals = compute_shap(models['xgb'], X_test.iloc[:200], model_name='xgb')
shap.summary_plot(shap_vals, X_test.iloc[:200], plot_type='bar', max_display=20)

## 9. Signal Generation

In [ ]:
from src.signals import generate_signals

signals = generate_signals(feat_df.tail(60), models, feature_cols, cfg, ticker=ticker)
buys = signals[signals['signal']=='BUY']
print(f'BUY signals in last 60 days: {len(buys)}')
buys.tail(10)